# Byte-Level BPE vs Character-Level BPE

### A From-Scratch Comparison

Byte Pair Encoding (BPE) is a subword tokenization algorithm used in many
modern language models.

This notebook implements BPE from scratch and compares two approaches:

1. **Character-Level BPE** — starts with characters as atomic symbols.
2. **Byte-Level BPE** — starts with UTF-8 bytes as atomic symbols.

The key idea is:

> **The BPE merging algorithm is the same. What changes is the initial representation of text.**

## 1. What We Will Learn

By the end of this notebook, we will understand:

- How BPE builds a vocabulary.
- How adjacent pair frequencies are calculated.
- How BPE learns merge rules.
- Why BPE can stop before complete words are formed.
- What the end-of-word marker `</w>` means.
- The difference between characters and UTF-8 bytes.
- How character-level BPE works.
- How byte-level BPE works.
- Why byte-level BPE can represent arbitrary Unicode text.
- How BPE encoding and decoding work.

## 2. What Is Byte Pair Encoding (BPE)?

BPE starts with a vocabulary of small atomic symbols and repeatedly combines
frequently occurring adjacent symbols.

For example, suppose we start with:

    l o w </w>

If the pair `(l, o)` is frequent, BPE may learn:

    l + o → lo

The word is now represented as:

    lo w </w>

If `(lo, w)` is later selected:

    lo + w → low

giving:

    low </w>

Each learned merge creates a new token.

Importantly, the old tokens are not removed from the vocabulary.

For example:

    Initial tokens: l, o, w, </w>
    Learned tokens: lo, low

The vocabulary can therefore contain both small and large tokens.

## 3. A Small BPE Training Example

We will first understand BPE using the classic small corpus:

| Word | Frequency |
|------|----------:|
| low | 5 |
| lower | 2 |
| newest | 6 |
| widest | 3 |

The frequency means that the word `low`, for example, occurs 5 times in the
training corpus.

We also use the special symbol:

    </w>

to represent the **end of a word**.

Therefore, before any BPE merges, the corpus is represented as:

    l o w </w>         × 5
    l o w e r </w>     × 2
    n e w e s t </w>   × 6
    w i d e s t </w>   × 3

In [3]:
# Training corpus: word -> frequency

word_freqs = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

word_freqs

{'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}

In [4]:
def initialize_corpus(word_freqs):
    """
    Split every word into characters and append </w>
    to mark the end of the word.
    """
    corpus = {}

    for word, freq in word_freqs.items():
        symbols = tuple(list(word) + ["</w>"])
        corpus[symbols] = freq

    return corpus


corpus = initialize_corpus(word_freqs)

for symbols, freq in corpus.items():
    print(" ".join(symbols), "×", freq)

l o w </w> × 5
l o w e r </w> × 2
n e w e s t </w> × 6
w i d e s t </w> × 3


### 3.1 Initial Vocabulary

Before BPE learns any merges, the vocabulary consists of the atomic symbols
present in the corpus.

For this example:

    l, o, w, e, r, n, s, t, i, d, </w>

BPE will add new tokens to this vocabulary as merges are learned.

In [6]:
def get_vocabulary(corpus):
    """Return all unique symbols currently present in the corpus."""
    vocab = set()

    for symbols in corpus:
        vocab.update(symbols)

    return vocab


initial_vocab = get_vocabulary(corpus)

print("Initial vocabulary:")
print(sorted(initial_vocab))

print("\nVocabulary size:", len(initial_vocab))

Initial vocabulary:
['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Vocabulary size: 11


## 4. Counting Adjacent Pairs

BPE counts how often every adjacent pair occurs.

Word frequencies must be taken into account.

For example:

    newest × 6
    widest × 3

Both contain the adjacent pair:

    e s

Therefore:

    count(e, s) = 6 + 3 = 9

Similarly:

    low   × 5
    lower × 2

both contain:

    l o

so:

    count(l, o) = 5 + 2 = 7

BPE selects one of the most frequent pairs for the next merge.

In [7]:
from collections import defaultdict


def get_pair_counts(corpus):
    """
    Count adjacent symbol pairs, weighted by word frequency.
    """
    pair_counts = defaultdict(int)

    for symbols, freq in corpus.items():

        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_counts[pair] += freq

    return dict(pair_counts)


pair_counts = get_pair_counts(corpus)

for pair, count in sorted(
    pair_counts.items(),
    key=lambda x: (-x[1], x[0])
):
    print(f"{pair}: {count}")

('e', 's'): 9
('s', 't'): 9
('t', '</w>'): 9
('w', 'e'): 8
('l', 'o'): 7
('o', 'w'): 7
('e', 'w'): 6
('n', 'e'): 6
('w', '</w>'): 5
('d', 'e'): 3
('i', 'd'): 3
('w', 'i'): 3
('e', 'r'): 2
('r', '</w>'): 2


### 4.1 Ties Between Pairs

Multiple pairs can have the same maximum frequency.

In our initial corpus:

    (e, s)      = 9
    (s, t)      = 9
    (t, </w>)   = 9

Therefore, there is a tie.

A BPE implementation needs a deterministic tie-breaking rule.

For this notebook, tied pairs are ordered lexicographically so that the
training process is reproducible.

Different valid tie-breaking rules may produce different merge orders.

In [8]:
def get_best_pair(corpus):
    """
    Return the most frequent adjacent pair.

    Lexicographic ordering is used to break frequency ties
    deterministically.
    """
    pair_counts = get_pair_counts(corpus)

    if not pair_counts:
        return None, 0

    best_pair = sorted(
        pair_counts.items(),
        key=lambda x: (-x[1], x[0])
    )[0]

    return best_pair


best_pair, frequency = get_best_pair(corpus)

print("Best pair:", best_pair)
print("Frequency:", frequency)

Best pair: ('e', 's')
Frequency: 9


## 5. Merging a Pair

Suppose BPE selects:

    e + s → es

Then:

    n e w e s t </w>

becomes:

    n e w es t </w>

and:

    w i d e s t </w>

becomes:

    w i d es t </w>

The newly created symbol `es` is now a BPE token.

In [9]:
def merge_pair_in_corpus(corpus, pair):
    """
    Replace every occurrence of a selected adjacent pair
    with its merged symbol.
    """
    merged_symbol = "".join(pair)
    new_corpus = {}

    for symbols, freq in corpus.items():

        new_symbols = []
        i = 0

        while i < len(symbols):

            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2

            else:
                new_symbols.append(symbols[i])
                i += 1

        new_corpus[tuple(new_symbols)] = freq

    return new_corpus

In [10]:
best_pair, frequency = get_best_pair(corpus)

print("Merging:", best_pair)
print("Frequency:", frequency)

corpus_after_one_merge = merge_pair_in_corpus(corpus, best_pair)

print("\nCorpus after merge:\n")

for symbols, freq in corpus_after_one_merge.items():
    print(" ".join(symbols), "×", freq)

Merging: ('e', 's')
Frequency: 9

Corpus after merge:

l o w </w> × 5
l o w e r </w> × 2
n e w es t </w> × 6
w i d es t </w> × 3


## 6. BPE Training

BPE repeats the same process:

1. Count all adjacent pairs.
2. Find the most frequent pair.
3. Merge that pair.
4. Add the resulting symbol to the vocabulary.
5. Recount the pairs.
6. Repeat.

The important point is that BPE does **not** have to continue until every word
becomes one token.

Training normally stops after reaching a chosen:

- number of merges, or
- target vocabulary size.

Therefore, BPE can intentionally stop "in between".

In [11]:
def train_bpe(corpus, num_merges, verbose=True):
    """
    Train BPE for a fixed number of merge operations.

    Returns:
        final_corpus
        merge_rules
        vocabulary
    """

    corpus = corpus.copy()

    # Vocabulary starts with the original atomic symbols.
    vocabulary = set()

    for symbols in corpus:
        vocabulary.update(symbols)

    merge_rules = []

    for step in range(num_merges):

        best_pair, frequency = get_best_pair(corpus)

        if best_pair is None:
            break

        new_token = "".join(best_pair)

        merge_rules.append(best_pair)
        vocabulary.add(new_token)

        corpus = merge_pair_in_corpus(corpus, best_pair)

        if verbose:
            print(
                f"Merge {step + 1}: "
                f"{best_pair[0]} + {best_pair[1]} "
                f"→ {new_token} "
                f"(frequency = {frequency})"
            )

    return corpus, merge_rules, vocabulary

In [12]:
corpus = initialize_corpus(word_freqs)

final_corpus, merge_rules, vocabulary = train_bpe(
    corpus,
    num_merges=8
)

Merge 1: e + s → es (frequency = 9)
Merge 2: es + t → est (frequency = 9)
Merge 3: est + </w> → est</w> (frequency = 9)
Merge 4: l + o → lo (frequency = 7)
Merge 5: lo + w → low (frequency = 7)
Merge 6: e + w → ew (frequency = 6)
Merge 7: ew + est</w> → ewest</w> (frequency = 6)
Merge 8: n + ewest</w> → newest</w> (frequency = 6)


In [13]:
print("Tokenization after training:\n")

for symbols, freq in final_corpus.items():
    print(f"{' | '.join(symbols):30s} × {freq}")

Tokenization after training:

low | </w>                     × 5
low | e | r | </w>             × 2
newest</w>                     × 6
w | i | d | est</w>            × 3


In [14]:
print("Learned merge rules:\n")

for i, pair in enumerate(merge_rules, start=1):
    print(
        f"{i:2d}. "
        f"{pair[0]} + {pair[1]} "
        f"→ {''.join(pair)}"
    )

Learned merge rules:

 1. e + s → es
 2. es + t → est
 3. est + </w> → est</w>
 4. l + o → lo
 5. lo + w → low
 6. e + w → ew
 7. ew + est</w> → ewest</w>
 8. n + ewest</w> → newest</w>


In [15]:
print("Final vocabulary:\n")

for token in sorted(vocabulary):
    print(repr(token))

print("\nVocabulary size:", len(vocabulary))

Final vocabulary:

'</w>'
'd'
'e'
'es'
'est'
'est</w>'
'ew'
'ewest</w>'
'i'
'l'
'lo'
'low'
'n'
'newest</w>'
'o'
'r'
's'
't'
'w'

Vocabulary size: 19


## 7. Vocabulary vs Tokenization

These are two different concepts.

### Vocabulary

The vocabulary contains:

- the original atomic symbols,
- tokens created by earlier merges,
- tokens created by later merges.

For example, it might contain:

    e
    s
    t
    es
    est
    est</w>

All of these can remain in the vocabulary.

### Tokenization

A particular word uses only the tokens produced by applying the learned merge
rules.

For example, depending on where training stops:

    widest → w | i | d | est</w>

Even though many other tokens exist in the vocabulary, they are not all used
for this word.

# Part II — Why Byte-Level BPE?

So far, our atomic symbols have been **characters**.

For example:

    hello → h | e | l | l | o

This is character-level BPE.

Byte-level BPE changes only the starting representation.

Instead of starting with characters, it starts with the bytes produced by
UTF-8 encoding.

The BPE merge algorithm itself remains the same.

## 8. UTF-8: Characters to Bytes

Computers represent text using numbers.

UTF stands for:

**Unicode Transformation Format**

UTF-8 represents Unicode characters using sequences of 8-bit units called
bytes.

One byte contains 8 bits.

ASCII characters require one byte in UTF-8:

    a → 1 byte

Other Unicode characters may require multiple bytes:

    é  → 2 bytes
    你 → 3 bytes
    😊 → 4 bytes

Therefore:

> One character is not always equal to one byte.

In [16]:
examples = ["a", "é", "你", "😊"]

for char in examples:
    encoded = char.encode("utf-8")

    print(
        repr(char),
        "→",
        list(encoded),
        f"→ {len(encoded)} byte(s)"
    )

'a' → [97] → 1 byte(s)
'é' → [195, 169] → 2 byte(s)
'你' → [228, 189, 160] → 3 byte(s)
'😊' → [240, 159, 152, 138] → 4 byte(s)


## 9. Character-Level vs Byte-Level Representation

Consider:

    Hi你

Character-level BPE initially sees:

    H | i | 你

There are 3 character symbols.

UTF-8 byte-level BPE sees:

    H  → 72
    i  → 105
    你 → 228 189 160

Therefore the initial byte sequence is:

    72 | 105 | 228 | 189 | 160

There are 5 byte symbols.

After this conversion, BPE performs the same repeated pair-merging algorithm
that we used earlier.

In [17]:
def text_to_byte_symbols(text):
    """
    Convert text into a tuple of UTF-8 byte values.
    """
    return tuple(text.encode("utf-8"))


text = "Hi你"

byte_symbols = text_to_byte_symbols(text)

print("Text:", text)
print("UTF-8 bytes:", byte_symbols)

Text: Hi你
UTF-8 bytes: (72, 105, 228, 189, 160)


## 10. Why Use Bytes?

A byte can have only 256 possible values:

    0, 1, 2, ..., 255

Therefore, byte-level BPE can begin with a fixed base vocabulary containing
all 256 byte values.

Any Unicode text can be converted into UTF-8 bytes.

This means that text containing:

- English
- Hindi
- Chinese
- Arabic
- emojis
- other Unicode characters

can always be represented as a sequence of bytes.

This avoids the fundamental unknown-character problem of a limited
character-level starting vocabulary.

In [19]:
byte_training_texts = {
    "hello": 5,
    "hello world": 3,
    "héllo": 2,
    "你好": 2,
}


def initialize_byte_corpus(text_freqs):
    """
    Convert each training text into UTF-8 byte symbols.
    """
    corpus = {}

    for text, freq in text_freqs.items():
        byte_symbols = tuple(text.encode("utf-8"))
        corpus[byte_symbols] = freq

    return corpus


byte_corpus = initialize_byte_corpus(byte_training_texts)

for symbols, freq in byte_corpus.items():
    print(symbols, "×", freq)

(104, 101, 108, 108, 111) × 5
(104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100) × 3
(104, 195, 169, 108, 108, 111) × 2
(228, 189, 160, 229, 165, 189) × 2


In [20]:
def initialize_byte_tokens(text_freqs):
    """
    Each atomic token is represented as a tuple containing one byte.

    Example:
        'a' -> (97,)
        'é' -> (195,), (169,)
    """
    corpus = {}

    for text, freq in text_freqs.items():

        byte_tokens = tuple(
            (byte_value,)
            for byte_value in text.encode("utf-8")
        )

        corpus[byte_tokens] = freq

    return corpus

In [21]:
def get_generic_pair_counts(corpus):
    """
    Count adjacent token pairs.

    Works for both character tokens and byte-sequence tokens.
    """
    pair_counts = defaultdict(int)

    for tokens, freq in corpus.items():

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_counts[pair] += freq

    return dict(pair_counts)

In [22]:
def merge_byte_pair(corpus, pair):
    """
    Merge two adjacent byte-sequence tokens.
    """

    merged_token = pair[0] + pair[1]
    new_corpus = {}

    for tokens, freq in corpus.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                new_tokens.append(merged_token)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        new_corpus[tuple(new_tokens)] = freq

    return new_corpus

In [23]:
def train_byte_bpe(corpus, num_merges, verbose=True):
    """
    Train byte-level BPE.

    Every token is represented internally as a tuple of byte values.
    """

    corpus = corpus.copy()

    vocabulary = {
        token
        for tokens in corpus
        for token in tokens
    }

    merge_rules = []

    for step in range(num_merges):

        pair_counts = get_generic_pair_counts(corpus)

        if not pair_counts:
            break

        best_pair, frequency = sorted(
            pair_counts.items(),
            key=lambda x: (-x[1], x[0])
        )[0]

        merged_token = best_pair[0] + best_pair[1]

        merge_rules.append(best_pair)
        vocabulary.add(merged_token)

        corpus = merge_byte_pair(corpus, best_pair)

        if verbose:
            print(
                f"Merge {step + 1}: "
                f"{best_pair} → {merged_token} "
                f"(frequency = {frequency})"
            )

    return corpus, merge_rules, vocabulary

In [24]:
byte_corpus = initialize_byte_tokens(byte_training_texts)

byte_final_corpus, byte_merges, byte_vocab = train_byte_bpe(
    byte_corpus,
    num_merges=10
)

Merge 1: ((108,), (108,)) → (108, 108) (frequency = 10)
Merge 2: ((108, 108), (111,)) → (108, 108, 111) (frequency = 10)
Merge 3: ((101,), (108, 108, 111)) → (101, 108, 108, 111) (frequency = 8)
Merge 4: ((104,), (101, 108, 108, 111)) → (104, 101, 108, 108, 111) (frequency = 8)
Merge 5: ((32,), (119,)) → (32, 119) (frequency = 3)
Merge 6: ((32, 119), (111,)) → (32, 119, 111) (frequency = 3)
Merge 7: ((32, 119, 111), (114,)) → (32, 119, 111, 114) (frequency = 3)
Merge 8: ((32, 119, 111, 114), (108,)) → (32, 119, 111, 114, 108) (frequency = 3)
Merge 9: ((32, 119, 111, 114, 108), (100,)) → (32, 119, 111, 114, 108, 100) (frequency = 3)
Merge 10: ((104, 101, 108, 108, 111), (32, 119, 111, 114, 108, 100)) → (104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100) (frequency = 3)


## 11. BPE Tokens Do Not Have to Match Character Boundaries

Byte-level BPE operates on a sequence of bytes.

For example:

    a → [97]
    é → [195, 169]

The text:

    aé

becomes:

    97 | 195 | 169

BPE learns merges based on frequent adjacent byte sequences.

Therefore, an intermediate BPE token does not necessarily have to correspond
to exactly one Unicode character.

What matters is that every token represents an exact sequence of bytes.

When decoding, the byte sequences are joined together and then decoded using
UTF-8.

In [25]:
def decode_byte_tokens(tokens):
    """
    Convert byte-level BPE tokens back into text.
    """

    all_bytes = []

    for token in tokens:
        all_bytes.extend(token)

    return bytes(all_bytes).decode("utf-8")


example_tokens = [
    (72,),
    (105,),
    (228, 189, 160),
]

decoded = decode_byte_tokens(example_tokens)

print(decoded)

Hi你


## 12. The Complete Byte-Level BPE Pipeline

### Encoding

    Text
      ↓
    UTF-8
      ↓
    Bytes
      ↓
    Apply learned BPE merges
      ↓
    BPE tokens
      ↓
    Token IDs

### Decoding

    Token IDs
      ↓
    BPE tokens
      ↓
    Byte sequences
      ↓
    Combine bytes
      ↓
    UTF-8 decoding
      ↓
    Original text

Each BPE token therefore has a mapping to an exact byte sequence.

## 13. Character-Level BPE vs Byte-Level BPE

| Property | Character-Level BPE | Byte-Level BPE |
|---|---|---|
| Initial units | Characters | UTF-8 bytes |
| BPE algorithm | Pair merging | Pair merging |
| Merge rules learned | Yes | Yes |
| Vocabulary grows through merges | Yes | Yes |
| Unicode handling | Depends on starting characters | Any UTF-8 text |
| Base alphabet | Potentially very large | 256 byte values |
| Character boundaries preserved | Yes | Not necessarily |

The important result is:

> **Byte-level BPE does not introduce a fundamentally different BPE algorithm.**

Both approaches repeatedly:

    count pairs → select frequent pair → merge → add token → repeat

The main difference is the representation used before BPE begins.

# 14. Key Takeaways

1. BPE begins with a set of atomic symbols.

2. It repeatedly merges frequent adjacent pairs.

3. Every merge creates a new vocabulary token.

4. Earlier/base tokens remain available in the vocabulary.

5. BPE can stop before complete words are formed.

6. Character-level BPE starts from Unicode characters.

7. Byte-level BPE starts from UTF-8 bytes.

8. UTF-8 characters may require 1–4 bytes.

9. There are only 256 possible byte values.

10. Byte-level BPE can therefore represent arbitrary UTF-8 text.

11. A byte-level BPE token represents an exact sequence of bytes.

12. The core BPE merging algorithm is the same in both approaches.

# Byte-Level BPE with Hindi — From Scratch

This section demonstrates Byte-Level BPE using Hindi text.

Unlike character-level BPE, Byte-Level BPE first converts text into UTF-8 bytes.
BPE then learns frequent patterns by merging adjacent byte sequences.

Pipeline:

    Hindi Text → UTF-8 Bytes → BPE Merges → Tokens

For decoding:

    Tokens → Byte Sequences → Concatenate Bytes → UTF-8 Decode → Hindi Text

The BPE merging algorithm itself remains unchanged; only the initial representation
uses bytes instead of characters.

In [26]:
examples = ["a", "न", "म", "भारत", "नमस्ते"]

for text in examples:
    encoded = text.encode("utf-8")

    print(f"Text: {text}")
    print(f"UTF-8 bytes: {list(encoded)}")
    print(f"Number of bytes: {len(encoded)}")
    print()

Text: a
UTF-8 bytes: [97]
Number of bytes: 1

Text: न
UTF-8 bytes: [224, 164, 168]
Number of bytes: 3

Text: म
UTF-8 bytes: [224, 164, 174]
Number of bytes: 3

Text: भारत
UTF-8 bytes: [224, 164, 173, 224, 164, 190, 224, 164, 176, 224, 164, 164]
Number of bytes: 12

Text: नमस्ते
UTF-8 bytes: [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]
Number of bytes: 18



In [27]:
hindi_word_freqs = {
    "नमस्ते": 10,
    "नमस्कार": 7,
    "भारत": 6,
    "भारतीय": 4,
}

print("Training Corpus:\n")

for word, freq in hindi_word_freqs.items():
    print(f"{word} × {freq}")

Training Corpus:

नमस्ते × 10
नमस्कार × 7
भारत × 6
भारतीय × 4


In [28]:
print("UTF-8 Representation:\n")

for word, freq in hindi_word_freqs.items():
    utf8_bytes = list(word.encode("utf-8"))

    print(f"{word}")
    print(f"Bytes: {utf8_bytes}")
    print(f"Frequency: {freq}")
    print()

UTF-8 Representation:

नमस्ते
Bytes: [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]
Frequency: 10

नमस्कार
Bytes: [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 149, 224, 164, 190, 224, 164, 176]
Frequency: 7

भारत
Bytes: [224, 164, 173, 224, 164, 190, 224, 164, 176, 224, 164, 164]
Frequency: 6

भारतीय
Bytes: [224, 164, 173, 224, 164, 190, 224, 164, 176, 224, 164, 164, 224, 165, 128, 224, 164, 175]
Frequency: 4



In [29]:
def initialize_byte_corpus(word_freqs):

    corpus = {}

    for word, freq in word_freqs.items():

        byte_tokens = tuple(
            (byte_value,)
            for byte_value in word.encode("utf-8")
        )

        corpus[byte_tokens] = freq

    return corpus


byte_corpus = initialize_byte_corpus(hindi_word_freqs)

for tokens, freq in byte_corpus.items():
    print(tokens, "×", freq)

((224,), (164,), (168,), (224,), (164,), (174,), (224,), (164,), (184,), (224,), (165,), (141,), (224,), (164,), (164,), (224,), (165,), (135,)) × 10
((224,), (164,), (168,), (224,), (164,), (174,), (224,), (164,), (184,), (224,), (165,), (141,), (224,), (164,), (149,), (224,), (164,), (190,), (224,), (164,), (176,)) × 7
((224,), (164,), (173,), (224,), (164,), (190,), (224,), (164,), (176,), (224,), (164,), (164,)) × 6
((224,), (164,), (173,), (224,), (164,), (190,), (224,), (164,), (176,), (224,), (164,), (164,), (224,), (165,), (128,), (224,), (164,), (175,)) × 4


In [30]:
from collections import defaultdict

def get_pair_counts(corpus):
    pair_counts = defaultdict(int)

    for tokens, freq in corpus.items():
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_counts[pair] += freq

    return dict(pair_counts)


pair_counts = get_pair_counts(byte_corpus)

In [31]:
sorted_pairs = sorted(
    pair_counts.items(),
    key=lambda x: (-x[1], x[0])
)

print("Most Frequent Byte Pairs:\n")

for pair, count in sorted_pairs[:15]:
    print(f"{pair} → {count}")

Most Frequent Byte Pairs:

((224,), (164,)) → 126
((224,), (165,)) → 31
((164,), (164,)) → 20
((141,), (224,)) → 17
((164,), (168,)) → 17
((164,), (174,)) → 17
((164,), (176,)) → 17
((164,), (184,)) → 17
((164,), (190,)) → 17
((165,), (141,)) → 17
((168,), (224,)) → 17
((174,), (224,)) → 17
((184,), (224,)) → 17
((190,), (224,)) → 17
((164,), (224,)) → 14


In [32]:
def get_best_pair(corpus):

    pair_counts = get_pair_counts(corpus)

    if not pair_counts:
        return None, 0

    best_pair, frequency = sorted(
        pair_counts.items(),
        key=lambda x: (-x[1], x[0])
    )[0]

    return best_pair, frequency


best_pair, frequency = get_best_pair(byte_corpus)

print("Best pair:", best_pair)
print("Frequency:", frequency)

Best pair: ((224,), (164,))
Frequency: 126


In [33]:
def merge_byte_pair(corpus, pair):

    # Example:
    # (224,) + (164,) -> (224, 164)
    merged_token = pair[0] + pair[1]

    new_corpus = {}

    for tokens, freq in corpus.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):
                new_tokens.append(merged_token)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        new_corpus[tuple(new_tokens)] = freq

    return new_corpus

In [34]:
best_pair, frequency = get_best_pair(byte_corpus)

print("Merging:")
print(best_pair[0], "+", best_pair[1])
print("Frequency:", frequency)

byte_corpus_after_merge = merge_byte_pair(
    byte_corpus,
    best_pair
)

print("\nCorpus after one BPE merge:\n")

for tokens, freq in byte_corpus_after_merge.items():
    print(tokens, "×", freq)

Merging:
(224,) + (164,)
Frequency: 126

Corpus after one BPE merge:

((224, 164), (168,), (224, 164), (174,), (224, 164), (184,), (224,), (165,), (141,), (224, 164), (164,), (224,), (165,), (135,)) × 10
((224, 164), (168,), (224, 164), (174,), (224, 164), (184,), (224,), (165,), (141,), (224, 164), (149,), (224, 164), (190,), (224, 164), (176,)) × 7
((224, 164), (173,), (224, 164), (190,), (224, 164), (176,), (224, 164), (164,)) × 6
((224, 164), (173,), (224, 164), (190,), (224, 164), (176,), (224, 164), (164,), (224,), (165,), (128,), (224, 164), (175,)) × 4


In [35]:
def train_byte_bpe(corpus, num_merges):

    corpus = corpus.copy()
    merge_rules = []

    for step in range(num_merges):

        best_pair, frequency = get_best_pair(corpus)

        if best_pair is None:
            break

        merged_token = best_pair[0] + best_pair[1]

        # Store the learned merge rule
        merge_rules.append(best_pair)

        # Apply the merge
        corpus = merge_byte_pair(corpus, best_pair)

        print(
            f"Merge {step + 1}: "
            f"{best_pair[0]} + {best_pair[1]} "
            f"→ {merged_token} "
            f"(frequency = {frequency})"
        )

    return corpus, merge_rules

In [36]:
byte_corpus = initialize_byte_corpus(hindi_word_freqs)

final_byte_corpus, merge_rules = train_byte_bpe(
    byte_corpus,
    num_merges=10
)

Merge 1: (224,) + (164,) → (224, 164) (frequency = 126)
Merge 2: (224,) + (165,) → (224, 165) (frequency = 31)
Merge 3: (224, 164) + (164,) → (224, 164, 164) (frequency = 20)
Merge 4: (168,) + (224, 164) → (168, 224, 164) (frequency = 17)
Merge 5: (168, 224, 164) + (174,) → (168, 224, 164, 174) (frequency = 17)
Merge 6: (168, 224, 164, 174) + (224, 164) → (168, 224, 164, 174, 224, 164) (frequency = 17)
Merge 7: (168, 224, 164, 174, 224, 164) + (184,) → (168, 224, 164, 174, 224, 164, 184) (frequency = 17)
Merge 8: (168, 224, 164, 174, 224, 164, 184) + (224, 165) → (168, 224, 164, 174, 224, 164, 184, 224, 165) (frequency = 17)
Merge 9: (168, 224, 164, 174, 224, 164, 184, 224, 165) + (141,) → (168, 224, 164, 174, 224, 164, 184, 224, 165, 141) (frequency = 17)
Merge 10: (190,) + (224, 164) → (190, 224, 164) (frequency = 17)


In [37]:
print("Learned BPE Merge Rules:\n")

for i, pair in enumerate(merge_rules, start=1):

    merged = pair[0] + pair[1]

    print(
        f"{i}. {pair[0]} + {pair[1]} → {merged}"
    )

Learned BPE Merge Rules:

1. (224,) + (164,) → (224, 164)
2. (224,) + (165,) → (224, 165)
3. (224, 164) + (164,) → (224, 164, 164)
4. (168,) + (224, 164) → (168, 224, 164)
5. (168, 224, 164) + (174,) → (168, 224, 164, 174)
6. (168, 224, 164, 174) + (224, 164) → (168, 224, 164, 174, 224, 164)
7. (168, 224, 164, 174, 224, 164) + (184,) → (168, 224, 164, 174, 224, 164, 184)
8. (168, 224, 164, 174, 224, 164, 184) + (224, 165) → (168, 224, 164, 174, 224, 164, 184, 224, 165)
9. (168, 224, 164, 174, 224, 164, 184, 224, 165) + (141,) → (168, 224, 164, 174, 224, 164, 184, 224, 165, 141)
10. (190,) + (224, 164) → (190, 224, 164)


In [38]:
print("Final Byte-Level Tokenization:\n")

for tokens, freq in final_byte_corpus.items():

    print("Tokens:")

    for token in tokens:
        print(token, end=" | ")

    print(f"\nFrequency: {freq}")
    print()

Final Byte-Level Tokenization:

Tokens:
(224, 164) | (168, 224, 164, 174, 224, 164, 184, 224, 165, 141) | (224, 164, 164) | (224, 165) | (135,) | 
Frequency: 10

Tokens:
(224, 164) | (168, 224, 164, 174, 224, 164, 184, 224, 165, 141) | (224, 164) | (149,) | (224, 164) | (190, 224, 164) | (176,) | 
Frequency: 7

Tokens:
(224, 164) | (173,) | (224, 164) | (190, 224, 164) | (176,) | (224, 164, 164) | 
Frequency: 6

Tokens:
(224, 164) | (173,) | (224, 164) | (190, 224, 164) | (176,) | (224, 164, 164) | (224, 165) | (128,) | (224, 164) | (175,) | 
Frequency: 4



In [39]:
def apply_merge_to_tokens(tokens, pair):

    merged_token = pair[0] + pair[1]

    new_tokens = []
    i = 0

    while i < len(tokens):

        if (
            i < len(tokens) - 1
            and tokens[i] == pair[0]
            and tokens[i + 1] == pair[1]
        ):
            new_tokens.append(merged_token)
            i += 2

        else:
            new_tokens.append(tokens[i])
            i += 1

    return tuple(new_tokens)

In [40]:
def encode_byte_bpe(text, merge_rules):

    # Step 1: Text → UTF-8 bytes
    utf8_bytes = text.encode("utf-8")

    # Step 2: Every byte initially becomes one token
    tokens = tuple(
        (byte_value,)
        for byte_value in utf8_bytes
    )

    # Step 3: Apply learned BPE rules in order
    for pair in merge_rules:
        tokens = apply_merge_to_tokens(tokens, pair)

    return tokens

In [41]:
text = "नमस्ते"

encoded_tokens = encode_byte_bpe(
    text,
    merge_rules
)

print("Original text:")
print(text)

print("\nUTF-8 bytes:")
print(list(text.encode("utf-8")))

print("\nBPE tokens:")
for token in encoded_tokens:
    print(token)

Original text:
नमस्ते

UTF-8 bytes:
[224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]

BPE tokens:
(224, 164)
(168, 224, 164, 174, 224, 164, 184, 224, 165, 141)
(224, 164, 164)
(224, 165)
(135,)


In [42]:
def decode_byte_bpe(tokens):

    all_bytes = []

    # Take the bytes stored inside every BPE token
    for token in tokens:
        all_bytes.extend(token)

    # Convert byte values into a bytes object
    byte_sequence = bytes(all_bytes)

    # UTF-8 decode
    text = byte_sequence.decode("utf-8")

    return text

In [43]:
decoded_text = decode_byte_bpe(encoded_tokens)

print("BPE tokens:")
print(encoded_tokens)

print("\nDecoded text:")
print(decoded_text)

BPE tokens:
((224, 164), (168, 224, 164, 174, 224, 164, 184, 224, 165, 141), (224, 164, 164), (224, 165), (135,))

Decoded text:
नमस्ते


In [44]:
print("1. BPE Tokens:")
print(encoded_tokens)


all_bytes = []

for token in encoded_tokens:
    all_bytes.extend(token)


print("\n2. Concatenated Bytes:")
print(all_bytes)


byte_sequence = bytes(all_bytes)

print("\n3. Bytes Object:")
print(byte_sequence)


decoded_text = byte_sequence.decode("utf-8")

print("\n4. UTF-8 Decoded Text:")
print(decoded_text)

1. BPE Tokens:
((224, 164), (168, 224, 164, 174, 224, 164, 184, 224, 165, 141), (224, 164, 164), (224, 165), (135,))

2. Concatenated Bytes:
[224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]

3. Bytes Object:
b'\xe0\xa4\xa8\xe0\xa4\xae\xe0\xa4\xb8\xe0\xa5\x8d\xe0\xa4\xa4\xe0\xa5\x87'

4. UTF-8 Decoded Text:
नमस्ते
